# multilingual-e5-large + UMAP + Gaussian Mixture Model

Pairs the finalized embedding model, `multilingual-e5-large`, with a **Gaussian Mixture Model** —
a soft/probabilistic alternative to K-Means (each article gets a probability of belonging to each
cluster; we take the most-likely cluster as its hard label for evaluation). Also benchmarked
alongside HDBSCAN/Agglomerative/K-Means in the news-event-detection paper (arXiv:2406.10552).

Sweeps `n_components` the same way the K-Means notebook sweeps k, keeping whichever value
maximizes silhouette score on the hard cluster assignments.

In [ ]:
!pip install -q umap-learn

In [ ]:
# Paths (Kaggle)
import pandas as pd
import numpy as np
import re
import unicodedata
from pathlib import Path

SEED = 42
np.random.seed(SEED)

DATA_DIR = Path("/kaggle/input/datasets/uom230425m/aggregation-pipeline-data/data")
RESULTS_DIR = Path("/kaggle/working/results/e5_gmm")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Data dir: {DATA_DIR.resolve()}")
print(f"Results dir: {RESULTS_DIR.resolve()}")

In [ ]:
# Load annotator files
df_a = pd.read_csv(DATA_DIR / "a.csv")
df_d = pd.read_csv(DATA_DIR / "d.csv")
df_p = pd.read_csv(DATA_DIR / "p.csv")

for frame in (df_a, df_d, df_p):
    frame.drop(columns=["bias_label"], inplace=True, errors="ignore")

print(df_a.shape, df_d.shape, df_p.shape)

In [ ]:
# Concatenate dataframes and drop duplicate articles by article_id
df = pd.concat([df_a, df_d, df_p], ignore_index=True, sort=False)
df = df.drop_duplicates(subset="article_id", keep="first").reset_index(drop=True)
df.drop(columns=["flags", "Unnamed: 8"], inplace=True, errors="ignore")

print("df shape:", df.shape)
df.head()

In [ ]:
# Keep rows with the fields required for clustering
required_columns = ["article_id", "publisher", "url", "published_at", "title", "body_text"]
df = df.dropna(subset=required_columns).copy()

print("df shape after required-field dropna:", df.shape)
df.info()

In [ ]:
# Unicode normalization (NFC - canonical decomposition + composition)
for column in ["title", "body_text"]:
    df[column] = df[column].astype(str).map(lambda value: unicodedata.normalize("NFC", value))

df.head()

In [ ]:
# Remove title duplication from the beginning of the body text
def remove_title_from_body(row):
    body = row["body_text"].strip()
    title = row["title"].strip()
    if body.startswith(title):
        body = body[len(title):].lstrip("\n").lstrip()
    return body


df["text"] = df.apply(remove_title_from_body, axis=1)
df[["title", "text"]].head()

In [ ]:
# Whitespace normalization
df["text"] = df["text"].map(lambda value: re.sub(r"\n{2,}", "\n", value))
df["text"] = df["text"].map(lambda value: re.sub(r"[ \t]+", " ", value))
df["text"] = df["text"].str.strip()

df = df[df["text"].str.len() > 0].reset_index(drop=True)
print("df shape after text cleaning:", df.shape)
df.head()

In [ ]:
# Build documents for embedding (title + body), with the "passage: " prefix multilingual-e5
# models require for corpus-side text
def build_passage_text(title, body):
    title = str(title).strip() if title else ""
    body = str(body).strip() if body else ""
    combined = f"{title}. {body}" if title else body
    return "passage: " + combined


df["passage_text"] = df.apply(lambda row: build_passage_text(row["title"], row["text"]), axis=1)
print(f"Documents: {len(df)}")
df["passage_text"].iloc[0][:500]

In [ ]:
# Load multilingual-e5-large (the finalized embedding model for this project)
import torch
from transformers import AutoTokenizer, AutoModel

device = "cuda" if torch.cuda.is_available() else "cpu"
embedding_model_name = "intfloat/multilingual-e5-large"
tokenizer = AutoTokenizer.from_pretrained(embedding_model_name)
model = AutoModel.from_pretrained(embedding_model_name).to(device)
model.eval()

In [ ]:
# Mean pooling — established as the best strategy for this model on this corpus in
# notebooks/clustering_e5.ipynb (separation_score 0.2130 for mean vs 0.1264 for max pooling)
def mean_pooling(last_hidden_state, attention_mask):
    mask = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
    summed = torch.sum(last_hidden_state * mask, dim=1)
    counts = torch.clamp(mask.sum(dim=1), min=1e-9)
    return summed / counts


@torch.no_grad()
def embed_passages(texts, batch_size=16, max_length=512):
    all_embeddings = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        encoded = tokenizer(
            batch, padding=True, truncation=True, max_length=max_length, return_tensors="pt",
        ).to(device)
        output = model(**encoded)
        pooled = mean_pooling(output.last_hidden_state, encoded["attention_mask"])
        pooled = torch.nn.functional.normalize(pooled, p=2, dim=1)
        all_embeddings.append(pooled.cpu())
    return torch.cat(all_embeddings, dim=0).numpy()


embeddings = embed_passages(df["passage_text"].tolist(), batch_size=16)
print(embeddings.shape)

In [ ]:
# Embedding separation score (comparable across notebooks): 1 - mean pairwise cosine similarity
# on a random 100-document sample of the raw embeddings.
from sklearn.metrics.pairwise import cosine_similarity

rng = np.random.default_rng(SEED)
sample_idx = rng.choice(len(embeddings), size=min(100, len(embeddings)), replace=False)
sample_emb = embeddings[sample_idx]

sims = cosine_similarity(sample_emb)
pairwise = sims[np.triu_indices_from(sims, k=1)]
separation_score = 1 - pairwise.mean()

print(f"Embedding model: {embedding_model_name}")
print(f"mean_sim={pairwise.mean():.4f} std={pairwise.std():.4f} min={pairwise.min():.4f} max={pairwise.max():.4f}")
print(f"Separation score (1 - mean cosine similarity): {separation_score:.4f}")

In [ ]:
# Reduce dimensionality with UMAP (same settings used across this project's UMAP-based
# notebooks, e.g. notebooks/clustering_BGE_M3_BERTopic.ipynb)
from umap import UMAP

umap_model = UMAP(
    n_neighbors=3,
    n_components=5,
    min_dist=0.0,
    metric="cosine",
    random_state=SEED,
)
reduced_embeddings = umap_model.fit_transform(embeddings)
print(reduced_embeddings.shape)

In [ ]:
# Sweep n_components for the GMM, pick the value with the best silhouette.
# reg_covar is raised above sklearn\'s default (1e-6) because with 1999 points in 5-D, large k
# means many components end up with very few points assigned — their per-dimension variance can
# come out at ~0 ("singleton or collapsed samples"), which crashes GaussianMixture\'s covariance
# inversion. A larger reg_covar floors every component\'s variance so that doesn\'t happen. Even
# so, individual (data-dependent) k values can still fail to converge to a valid fit — those are
# skipped rather than crashing the whole sweep.
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score

REG_COVAR = 1e-3
candidate_k = sorted(set(list(range(10, 100, 10)) + list(range(100, 400, 25)) + list(range(400, 900, 50))))
sweep_results = []

for k in candidate_k:
    if k >= len(reduced_embeddings):
        continue
    try:
        gmm = GaussianMixture(n_components=k, random_state=SEED, covariance_type="diag", reg_covar=REG_COVAR)
        labels = gmm.fit_predict(reduced_embeddings)
    except ValueError as e:
        print(f"k={k:>4}  skipped (fit failed: {e})")
        continue
    n_found = len(set(labels))
    if n_found > 1:
        sil = silhouette_score(reduced_embeddings, labels, metric="cosine")
        sweep_results.append((k, sil))
        print(f"k={k:>4}  silhouette={sil:.4f}")

sweep_df = pd.DataFrame(sweep_results, columns=["k", "silhouette"]).sort_values("silhouette", ascending=False)
sweep_df.head(10)

In [ ]:
# Refit at the best k found (same reg_covar as the sweep, for consistency)
best_k = int(sweep_df.iloc[0]["k"])
print(f"Best k: {best_k}")

gmm = GaussianMixture(n_components=best_k, random_state=SEED, covariance_type="diag", reg_covar=REG_COVAR)
df["cluster_id"] = gmm.fit_predict(reduced_embeddings)

print(df["cluster_id"].value_counts().head(20))
print("Number of clusters found:", df["cluster_id"].nunique())

extra_row_fields = {"best_k": best_k, "reg_covar": REG_COVAR}

In [ ]:
# Clustering evaluation
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score

labels = df["cluster_id"].values
mask = labels != -1  # -1 = noise; only meaningful for density-based algorithms
X_valid = reduced_embeddings[mask]
labels_valid = labels[mask]
n_clusters = len(set(labels_valid))
noise_ratio = 1 - mask.mean()

print(f"Model: {embedding_model_name} + UMAP + GMM")
print(f"Articles: {len(df)} | Clusters (excl. noise): {n_clusters} | Noise ratio: {noise_ratio:.2%}")

if n_clusters > 1:
    sil = silhouette_score(X_valid, labels_valid, metric="cosine")
    dbi = davies_bouldin_score(X_valid, labels_valid)
    ch = calinski_harabasz_score(X_valid, labels_valid)
    print(f"Silhouette Score (cosine): {sil:.4f}")
    print(f"Davies-Bouldin Index:      {dbi:.4f}  (lower is better)")
    print(f"Calinski-Harabasz Index:   {ch:.2f}  (higher is better)")
else:
    sil = dbi = ch = float("nan")
    print("Not enough clusters to compute silhouette/DBI/CH.")

scores_path = RESULTS_DIR / "e5_gmm_scores.csv"
row = {
    "model": embedding_model_name,
    "pipeline": "UMAP+GMM",
    "embedding_dim": embeddings.shape[1],
    "separation_score": round(separation_score, 4),
    "n_articles": len(df),
    "n_clusters": n_clusters,
    "noise_ratio": round(noise_ratio, 4),
    "silhouette": round(sil, 4) if n_clusters > 1 else None,
    "davies_bouldin": round(dbi, 4) if n_clusters > 1 else None,
    "calinski_harabasz": round(ch, 2) if n_clusters > 1 else None,
}
row.update(extra_row_fields)
pd.DataFrame([row]).to_csv(scores_path, index=False)
print(f"Saved scores to {scores_path}")

In [ ]:
# Inspect sample titles per cluster
for cluster_id, group in list(df[df["cluster_id"] != -1].groupby("cluster_id"))[:10]:
    print(f"=== Cluster {cluster_id} ({len(group)} articles) ===")
    for title in group["title"].head(10):
        print(f"- {title}")
    print()

In [ ]:
# Inspect a RANDOM sample of clusters (rather than just the first few by id) — a more
# representative check of overall cluster quality than always looking at the same low-numbered
# clusters
rng_inspect = np.random.default_rng(SEED)
cluster_ids = df.loc[df["cluster_id"] != -1, "cluster_id"].unique()
sample_size = min(8, len(cluster_ids))
sampled_cluster_ids = rng_inspect.choice(cluster_ids, size=sample_size, replace=False)

for cluster_id in sampled_cluster_ids:
    group = df[df["cluster_id"] == cluster_id]
    print(f"=== Cluster {cluster_id} ({len(group)} articles) ===")
    for title in group["title"].head(15):
        print(f"- {title}")
    print()

In [ ]:
# Save article-level assignments
assignments_path = RESULTS_DIR / "e5_gmm_assignments.csv"
df.drop(columns=["passage_text"], errors="ignore").to_csv(assignments_path, index=False, encoding="utf-8-sig")
print(f"Saved assignments: {assignments_path.resolve()}")